In [ ]:
from collections import Counter

import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

1. 学習データ読み込み

In [ ]:
file_path = "../data/raw/train-00000-of-00001.parquet"
df_raw = pd.read_parquet(file_path)

2.前処理

In [ ]:
df_train = df_raw.copy()

In [ ]:
col_names_categorical = [
    "job", "marital", "education", "contact", "month", "poutcome"
]

col_names_binary = [
    "default", "housing", "loan", "y"
]

col_names_object = [
    "poutcome"
]

In [ ]:
# 前処理関数
# 学習データとテストデータで同一にする

def preprocessing(df):
    
    # カテゴリー型に変更
    for col in col_names_categorical:
        df[col] = df[col].astype("category")

    # binary(yes or no)のカラムをboolに変換
    for col in col_names_binary:
        df[col] = df[col].str.lower().map({"yes": True, "no": False})

    # object型をラベルエンコーディング
    for col in col_names_object:
        df[col], _ =  pd.factorize(df[col])

    return df

In [ ]:
df_train = preprocessing(df=df_train)

3.学習

3.1 データセット作成

In [ ]:
df_train.columns

In [ ]:
y_col = "y"

X = df_train.drop(y_col, axis=1)
y = df_train[y_col]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

train_dataset = lgb.Dataset(X_train, label=y_train)
val_dataset = lgb.Dataset(X_val, label=y_val, reference=train_dataset)


3.2 学習の実施

In [ ]:
params = {
    "objective": "binary",      # 2値分類
    "metric": "binary_logloss", # 評価指標
    "verbose": -1
}

callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=True)
]

In [ ]:
model = lgb.train(
    params,
    train_dataset,
    valid_sets=[val_dataset], 
    callbacks = callbacks
)


4.予測&評価

In [ ]:
file_path = "../data/raw/test-00000-of-00001.parquet"
df_test = pd.read_parquet(file_path)

In [ ]:
df_test = preprocessing(df=df_test)


In [ ]:
X_test = df_test.drop(y_col, axis=1)
y_test = df_test[y_col]

In [ ]:
y_pred_prob = model.predict(X_test)
y_pred = [1 if i >= 0.5 else 0 for i in y_pred_prob]

In [ ]:
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")


In [ ]:
# yが不均衡データなので、予測値の中身を確認
counts = Counter(y_pred)
print(counts)
print("yesの割合:", counts[1]/(counts[0]+counts[1]))

In [ ]:
# 実際のyのTrue, Falseを見る
df_test["y"].value_counts(normalize=True)

In [ ]:
print(f"precision_score: {precision_score(y_test, y_pred, pos_label=True)}")
print(f"recall_score: {recall_score(y_test, y_pred, pos_label=True)}")
print(f"f1_score: {f1_score(y_test, y_pred, pos_label=True)}")

# recallが0.56なので、実際にyがTrueのうち、しっかりとTrueと予測できた割合が56%で半分程度